# Flight duration model: More features!

Let's add more features to our model. This will not necessarily result in a better model. Adding some features might improve the model. Adding other features might make it worse.

More features will always make the model more complicated and difficult to interpret.

These are the features you'll include in the next model:  

- `km`
- `org` (origin airport, one-hot encoded, 8 levels)
- `depart` (departure time, binned in 3 hour intervals, one-hot encoded, 8 levels)
- `dow` (departure day of week, one-hot encoded, 7 levels) and
- `mon` (departure month, one-hot encoded, 12 levels).

These have been assembled into the `features` column, which is a sparse representation of 32 columns (remember one-hot encoding produces a number of columns which is one fewer than the number of levels).

The data are available as `flight`, randomly split into `flights_train` and `flights_test`.

This exercise is based on a small subset of the flights data.

## Instructions

- Fit a linear regression model to the training data.
- Generate predictions for the testing data.
- Calculate the RMSE on the testing data.
- Look at the model coefficients. Are any of them zero?

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [3]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M3-Regression/4_Regularization/dataset/flights-1000.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')

In [4]:
from pyspark.sql.functions import round
flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')

from pyspark.ml.feature import StringIndexer

flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

from pyspark.ml.feature import Bucketizer, OneHotEncoder, OneHotEncoderEstimator

# onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])

flights = onehot.fit(flights).transform(flights)

buckets = Bucketizer(splits=[0, 3, 6, 9, 12, 15, 18, 21, 24]\
                     , inputCol='depart', outputCol='depart_bucket')

flights = buckets.transform(flights)

# onehot = OneHotEncoder(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['depart_bucket'], outputCols=['depart_dummy'])
flights = onehot.fit(flights).transform(flights)

# onehot = OneHotEncoder(inputCols=['dow', 'mon'], outputCols=['dow_dummy', 'mon_dummy'])
onehot = OneHotEncoderEstimator(inputCols=['dow', 'mon']\
                                , outputCols=['dow_dummy', 'mon_dummy'])
flights = onehot.fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['km', 'org_dummy', 'depart_dummy', 'dow_dummy', 'mon_dummy'\
], outputCol='features')

flights = assembler.transform(flights)

print("Subset from the flights DataFrame:")

flights.select('features', 'duration').show(5, truncate=False)

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

Subset from the flights DataFrame:
+--------------------------------------------+--------+
|features                                    |duration|
+--------------------------------------------+--------+
|(32,[0,3,11],[3465.0,1.0,1.0])              |351     |
|(32,[0,1,13,17,21],[509.0,1.0,1.0,1.0,1.0]) |82      |
|(32,[0,2,10,19,23],[542.0,1.0,1.0,1.0,1.0]) |82      |
|(32,[0,1,11,16,30],[1989.0,1.0,1.0,1.0,1.0])|195     |
|(32,[0,1,10,20,25],[415.0,1.0,1.0,1.0,1.0]) |65      |
+--------------------------------------------+--------+
only showing top 5 rows



In [ ]:
from pyspark.ml.regression import ____
from pyspark.ml.evaluation import ____

# Fit linear regression model to training data
regression = ____(____).____(____)

# Make predictions on testing data
predictions = regression.____(____)

# Calculate the RMSE on testing data
rmse = ____(____).____(____)
print("The test RMSE is", rmse)

# Look at the model coefficients
coeffs = regression.____
print(coeffs)